In [28]:
# low-rank approximation of the epistatic operator Wu
import numpy as np

n = 500
m = 200
p = m * (m - 1) / 2
r = 199

rng = np.random.default_rng()

X = rng.binomial(2, 0.2, size=(n, m))
Z = (X - X.mean(axis=0)) / X.std(axis=0)
D = Z * Z

u = rng.normal(0, 1, size=n)

# ---- leading eigenpairs of K_w = Z Z^T, taken from Z ----
U, s, Pt = np.linalg.svd(Z, full_matrices=False)
Q = U[:, :r]                                 # n x r
lam = s[:r] ** 2                             # top-r eigenvalues of K_w

# ---- operator ----
G = Q * u[:, None]                           # n x r, columns q_s o u
KG = Z @ (Z.T @ G)                           # n x r, columns K (q_s o u)
Wu_lr = ((Q * KG) @ lam - D @ (D.T @ u)) / (2 * p)

# ---- exact reference ----
M = Z.T @ (u[:, None] * Z)
Wu_exact = ((Z * (Z @ M)) @ np.ones(m) - D @ (D.T @ u)) / (2 * p)

print(np.linalg.norm(Wu_lr - Wu_exact) / np.linalg.norm(Wu_exact))

0.001845505198706931


In [38]:
# compare lowrank Wu with unstd Wu for random Wu
import numpy as np

n = 1000
m = 500
p = m * (m - 1) / 2
r_list = [10, 25, 50, 100, 150, 200,500]

rng = np.random.default_rng()

X = rng.binomial(2, 0.2, size=(n, m))
Z = (X - X.mean(axis=0)) / X.std(axis=0)
D = Z * Z
u = rng.normal(0, 1, size=n)

# ---- exact reference, fixed throughout ----
j, k = np.triu_indices(m, k=1)
H = Z[:, j] * Z[:, k]
Wu_exact = H @ (H.T @ u) / p
Dterm = D @ (D.T @ u)

# ---- leading eigenpairs of K_w, taken from Z ----
U, s, Pt = np.linalg.svd(Z, full_matrices=False)

for r in r_list:
    Q = U[:, :r]
    lam = s[:r] ** 2
    G = Q * u[:, None]
    KG = Z @ (Z.T @ G)
    Wu_lr = ((Q * KG) @ lam - Dterm) / (2 * p)

    trace_kept = lam.sum() / (s ** 2).sum()
    mean_err = (Wu_lr - Wu_exact).mean()
    rel_L2 = np.linalg.norm(Wu_lr - Wu_exact) / np.linalg.norm(Wu_exact)
    print(f"r = {r:5d}  PC variance = {trace_kept:.4f}"
          f"  mean signed err = {mean_err:+.3e}  rel L2 norm err = {rel_L2:.4f}")

print("The first 5 elements of exact Wu and of low-rank Wu:")
print("Exact    ", np.round(Wu_exact[:5], 4))
print("Low-rank ", np.round(Wu_lr[:5], 4))

r =    10  PC variance = 0.0548  mean signed err = +9.137e-02  rel L2 norm err = 0.9525
r =    25  PC variance = 0.1299  mean signed err = +8.196e-02  rel L2 norm err = 0.8749
r =    50  PC variance = 0.2415  mean signed err = +6.372e-02  rel L2 norm err = 0.7611
r =   100  PC variance = 0.4276  mean signed err = +3.426e-02  rel L2 norm err = 0.5717
r =   150  PC variance = 0.5763  mean signed err = +2.065e-02  rel L2 norm err = 0.4223
r =   200  PC variance = 0.6953  mean signed err = +1.181e-02  rel L2 norm err = 0.3025
r =   500  PC variance = 1.0000  mean signed err = -9.880e-17  rel L2 norm err = 0.0000
The first 5 elements of exact Wu and of low-rank Wu:
Exact     [ 0.2503 -0.8524  0.5223 -0.2377  1.7517]
Low-rank  [ 0.2503 -0.8524  0.5223 -0.2377  1.7517]


In [50]:
# compare lowrank Wu with unstd Wu, tagged SNPs (rho = 0.9)
import numpy as np

n = 1000
m = 500
p = m * (m - 1) / 2
rho = 0.9
maf = 0.2
r_list = [10, 25, 50, 100, 150, 200, 500]

rng = np.random.default_rng()

# ---- all SNPs tag one causal haplotype, copied w.p. rho ----
def haplotypes():
    causal = rng.binomial(1, maf, size=(n, 1))
    h = rng.binomial(1, maf, size=(n, m))
    copy = rng.binomial(1, rho, size=(n, m))
    return np.where(copy, causal, h)

X = haplotypes() + haplotypes()
Z = (X - X.mean(axis=0)) / X.std(axis=0)
D = Z * Z
u = rng.normal(0, 1, size=n)

# ---- exact reference, fixed throughout ----
Dterm = D @ (D.T @ u)
M = Z.T @ (u[:, None] * Z)
Wu_exact = ((Z * (Z @ M)) @ np.ones(m) - Dterm) / (2 * p)

# ---- leading eigenpairs of K_w, taken from Z ----
U, s, Pt = np.linalg.svd(Z, full_matrices=False)

for r in r_list:
    Q = U[:, :r]
    lam = s[:r] ** 2
    G = Q * u[:, None]
    KG = Z @ (Z.T @ G)
    Wu_lr = ((Q * KG) @ lam - Dterm) / (2 * p)

    var_exp = lam.sum() / (s ** 2).sum()
    mean_err = (Wu_lr - Wu_exact).mean()
    rel_L2 = np.linalg.norm(Wu_lr - Wu_exact) / np.linalg.norm(Wu_exact)
    print(f"r = {r:5d}  var explained = {var_exp:.4f}"
          f"  mean signed err = {mean_err:+.3e}  rel L2 norm err = {rel_L2:.4f}")


r =    10  var explained = 0.8210  mean signed err = -5.417e-03  rel L2 norm err = 0.0051
r =    25  var explained = 0.8372  mean signed err = -3.720e-03  rel L2 norm err = 0.0035
r =    50  var explained = 0.8605  mean signed err = -2.284e-03  rel L2 norm err = 0.0028
r =   100  var explained = 0.8977  mean signed err = -1.109e-03  rel L2 norm err = 0.0017
r =   150  var explained = 0.9261  mean signed err = -5.215e-04  rel L2 norm err = 0.0011
r =   200  var explained = 0.9479  mean signed err = -2.469e-04  rel L2 norm err = 0.0007
r =   500  var explained = 1.0000  mean signed err = +4.220e-13  rel L2 norm err = 0.0000


In [49]:
# compare lowrank Wu with unstd Wu, fixed r n m, change with rho
import numpy as np

n = 1000
m = 500
p = m * (m - 1) / 2
r = 50
maf = 0.2
rho_list = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1]

rng = np.random.default_rng()

u = rng.normal(0, 1, size=n)

# ---- all SNPs tag one causal haplotype, copied w.p. rho ----
def haplotypes(rho):
    causal = rng.binomial(1, maf, size=(n, 1))
    h = rng.binomial(1, maf, size=(n, m))
    copy = rng.binomial(1, rho, size=(n, m))
    return np.where(copy, causal, h)

for rho in rho_list:
    X = haplotypes(rho) + haplotypes(rho)
    Z = (X - X.mean(axis=0)) / X.std(axis=0)
    D = Z * Z

    # ---- exact reference ----
    Dterm = D @ (D.T @ u)
    M = Z.T @ (u[:, None] * Z)
    Wu_exact = ((Z * (Z @ M)) @ np.ones(m) - Dterm) / (2 * p)

    # ---- leading eigenpairs of K_w, taken from Z ----
    U, s, Pt = np.linalg.svd(Z, full_matrices=False)
    Q = U[:, :r]
    lam = s[:r] ** 2
    G = Q * u[:, None]
    KG = Z @ (Z.T @ G)
    Wu_lr = ((Q * KG) @ lam - Dterm) / (2 * p)

    var_exp = lam.sum() / (s ** 2).sum()
    mean_err = (Wu_lr - Wu_exact).mean()
    rel_L2 = np.linalg.norm(Wu_lr - Wu_exact) / np.linalg.norm(Wu_exact)
    print(f"rho = {rho:5.2f}  var explained = {var_exp:.4f}"
          f"  mean signed err = {mean_err:+.3e}  rel L2 norm err = {rel_L2:.4f}")


rho =  0.00  var explained = 0.2405  mean signed err = +1.448e-02  rel L2 norm err = 0.7599
rho =  0.30  var explained = 0.3098  mean signed err = +1.497e-02  rel L2 norm err = 0.6287
rho =  0.50  var explained = 0.4383  mean signed err = +1.982e-02  rel L2 norm err = 0.3028
rho =  0.70  var explained = 0.6282  mean signed err = +4.961e-03  rel L2 norm err = 0.0261
rho =  0.90  var explained = 0.8627  mean signed err = +3.755e-04  rel L2 norm err = 0.0086
rho =  0.99  var explained = 0.9863  mean signed err = -7.279e-06  rel L2 norm err = 0.0017
rho =  1.00  var explained = 1.0000  mean signed err = +4.869e-14  rel L2 norm err = 0.0000
